In [5]:
# Integrated AI-Powered pKₐ Prediction Notebook (Updated)
# -----------------------------------------------------------
# This code loads the dataset, computes molecular descriptors,
# builds a machine learning model (XGBoost in this example),
# and provides an interactive widget to predict pKₐ values from an IUPAC name.
#
# Requirements:
# - pandas, numpy, scikit-learn, xgboost, rdkit, requests, ipywidgets
#
# Make sure the dataset "iupac_high-confidence_v2_2.csv" is in the same directory.
# -----------------------------------------------------------

import pandas as pd
import numpy as np
import requests
from urllib.parse import quote
from rdkit import Chem
from rdkit.Chem import Descriptors, AllChem
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import r2_score, mean_squared_error
from xgboost import XGBRegressor
from sklearn.preprocessing import StandardScaler
import ipywidgets as widgets
from IPython.display import display

# --- Helper Functions ---
def name_to_smiles(name):
    """
    Converts an IUPAC name to SMILES using the NCI Chemical Identifier Resolver.
    """
    url = f"http://cactus.nci.nih.gov/chemical/structure/{quote(name)}/smiles"
    try:
        response = requests.get(url)
        if response.status_code == 200:
            return response.text.strip()
    except Exception as e:
        print("Error in name_to_smiles:", e)
    return None

def compute_features(smiles):
    """
    Computes basic RDKit descriptors for a given SMILES string.
    Returns a dictionary of features.
    """
    mol = Chem.MolFromSmiles(smiles)
    feats = {}
    if mol:
        feats['MolWt']   = Descriptors.MolWt(mol)
        feats['LogP']    = Descriptors.MolLogP(mol)
        feats['NumHBD']  = Descriptors.NumHDonors(mol)
        feats['NumHBA']  = Descriptors.NumHAcceptors(mol)
        feats['TPSA']    = Descriptors.TPSA(mol)
    else:
        feats['MolWt'] = feats['LogP'] = feats['NumHBD'] = feats['NumHBA'] = feats['TPSA'] = np.nan
    return feats

# --- 1. Load and Explore the Dataset ---
print("Loading dataset...")
df = pd.read_csv('iupac_high-confidence_v2_2.csv')
print("Total entries:", len(df))

# Replace missing solvents with 'water'
df['solvent'] = df['solvent'].fillna('water')

# Ensure the temperature column 'T' is numeric and fill missing values with 25.0.
if 'T' not in df.columns:
    df['T'] = 25.0
else:
    df['T'] = pd.to_numeric(df['T'], errors='coerce')
    df['T'] = df['T'].fillna(25.0)

# --- 2. Feature Engineering ---
print("Computing molecular descriptors...")
desc_list = df['SMILES'].apply(compute_features)
desc_df = pd.DataFrame(list(desc_list))
print("Descriptors computed for", len(desc_df), "molecules.")

# One-hot encode the 'solvent' column
solvent_dummies = pd.get_dummies(df['solvent'], prefix='solvent')

# Combine descriptors with solvent one-hot columns and temperature
X_desc = pd.concat([desc_df, solvent_dummies, df[['T']]], axis=1)

# Prepare the target variable: pKa values (convert to numeric and drop NaNs)
y = pd.to_numeric(df['pka_value'], errors='coerce')
mask = ~y.isna()
X_desc = X_desc[mask]
y = y[mask]
print("Dataset prepared with", X_desc.shape[0], "entries and", X_desc.shape[1], "features.")

# --- 3. Train/Test Split ---
X_train, X_test, y_train, y_test = train_test_split(X_desc, y, test_size=0.2, random_state=42)
print("Training samples:", len(X_train), "| Test samples:", len(X_test))

# --- 4. Model Training with XGBoost ---
print("Training XGBoost model...")
xgb_model = XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42)
xgb_model.fit(X_train, y_train)

# --- 5. Model Evaluation ---
y_pred = xgb_model.predict(X_test)
r2 = r2_score(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred, squared=False)
print(f"Initial XGBoost Model: R² = {r2:.3f}, RMSE = {rmse:.2f}")

# For simplicity, we'll use the initial xgb_model as our best model.
best_model = xgb_model

# --- 6. User Interface for pKₐ Prediction ---
# Define the expected feature order:
# ['MolWt', 'LogP', 'NumHBD', 'NumHBA', 'TPSA'] + list(solvent_dummies.columns) + ['T']
# For user input, we assume water as the default solvent and T = 25.

name_input = widgets.Text(
    description="IUPAC Name:",
    placeholder="Enter chemical name"
)
predict_button = widgets.Button(description="Predict pKₐ")
output_label = widgets.Label(value="")

def on_predict_clicked(b):
    name = name_input.value.strip()
    if not name:
        output_label.value = "Please enter a chemical name."
        return
    
    # Convert name to SMILES
    smiles = name_to_smiles(name)
    if not smiles:
        output_label.value = "Name not recognized. Please check spelling."
        return
    
    # Compute descriptors for the given SMILES
    feats = compute_features(smiles)
    if any(np.isnan(list(feats.values()))):
        output_label.value = "Error computing descriptors for the molecule."
        return
    
    # One-hot encode solvent: default to water
    sol_features = {col: 0 for col in solvent_dummies.columns}
    water_col = [col for col in sol_features.keys() if 'water' in col.lower()]
    if water_col:
        sol_features[water_col[0]] = 1
    else:
        sol_features['solvent_water'] = 1
    
    # Build the feature vector in the same order as used in training:
    # Order: ['MolWt', 'LogP', 'NumHBD', 'NumHBA', 'TPSA']
    descriptor_vals = [feats['MolWt'], feats['LogP'], feats['NumHBD'], feats['NumHBA'], feats['TPSA']]
    # Add solvent one-hot values (ensuring order matches training data)
    solvent_order = list(solvent_dummies.columns)
    solvent_vals = [sol_features.get(col, 0) for col in solvent_order]
    # Add temperature, assume 25 if not specified
    temp_val = 25.0
    # Combine all features
    feature_vector = np.array(descriptor_vals + solvent_vals + [temp_val]).reshape(1, -1)
    
    # Predict pKₐ
    pred_pka = best_model.predict(feature_vector)[0]
    output_label.value = f"Predicted pKₐ: {pred_pka:.2f}"

predict_button.on_click(on_predict_clicked)

# Display the interactive widget
display(name_input, predict_button, output_label)


Loading dataset...
Total entries: 24222
Computing molecular descriptors...
Descriptors computed for 24222 molecules.
Dataset prepared with 24028 entries and 131 features.
Training samples: 19222 | Test samples: 4806
Training XGBoost model...
Initial XGBoost Model: R² = 0.449, RMSE = 2.98


C:\Users\Fahad\anaconda3\Lib\site-packages\sklearn\metrics\_regression.py:483: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(


Text(value='', description='IUPAC Name:', placeholder='Enter chemical name')

Button(description='Predict pKₐ', style=ButtonStyle())

Label(value='')